# 06 · Awards: You vs the Machine

Predict the season's awards from a point in the middle of the season (`AFTER_GW`). Each correct award is worth **5 points**.

| Award | How the winner is decided here |
|---|---|
| Player of the Week | most FPL points in a gameweek |
| Player of the Season | most FPL points over the season |
| Golden Boot | most goals |
| Playmaker | most assists (FPL counts assists slightly differently from the official award) |
| Golden Glove | goalkeeper with the most clean sheets |

The real Player of the Season is chosen by a vote, and there have only been a handful of winners, which is far too few to
train a model on. So here it means "best FPL season", which can be measured.

**How the machine projects the rest of the season** (`pl.award_projections`):
- *so far* + *rate per team match so far* × *remaining fixtures*
- goals and assists are scaled by how many goals the team is expected to score in its remaining fixtures (Poisson model)
- clean sheets add up P(clean sheet) = e^(−expected goals conceded) over the remaining fixtures, times how often the keeper plays
- team form and Elo are frozen at `AFTER_GW`, so no future results are used

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))   # run from notebooks/, so "src" is one folder up
from src import pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)

In [2]:
# ---- settings ----
SEASON = "2025-26"
AFTER_GW = 19          # make the predictions after this gameweek (19 = halfway)
PLAYER = "You"

In [3]:
mdf, team_games, elo_hist = pl.build_match_table()
lam = pl.walk_forward_lambdas(mdf)
fpl = pl.load_fpl()
pgw = pl.build_player_gameweeks(pl.add_fixture_context(fpl, lam))
fixtures = pl.fpl_fixtures(fpl).merge(mdf[["Season", "Home_Team", "Away_Team", "No_Crowd"]],
                                      on=["Season", "Home_Team", "Away_Team"])

goal_models = pl.fit_goal_models(mdf[mdf["Season"] < SEASON])
proj = pl.award_projections(pgw, fixtures, team_games, elo_hist, goal_models, SEASON, AFTER_GW)

cols = {"Player of the Season": ("Points", "Proj_Points"), "Golden Boot": ("Goals", "Proj_Goals"),
        "Playmaker": ("Assists", "Proj_Assists"), "Golden Glove": ("Clean_Sheets", "Proj_Clean_Sheets")}
for award, (so_far, projected) in cols.items():
    t = proj.sort_values(projected, ascending=False).head(5)[["Name", "Team", so_far, projected]].round(1)
    print(f"--- {award}: top 5 projected after GW {AFTER_GW}")
    print(t.to_string(index=False))

--- Player of the Season: top 5 projected after GW 19
                        Name        Team  Points  Proj_Points
              Erling Haaland    Man City     155        310.0
             Antoine Semenyo Bournemouth     113        226.0
                 Declan Rice     Arsenal     102        204.0
      Bruno Borges Fernandes  Man United      99        198.0
Gabriel dos Santos Magalhães     Arsenal      97        194.0
--- Golden Boot: top 5 projected after GW 19
                            Name        Team  Goals  Proj_Goals
                  Erling Haaland    Man City     19        45.3
Igor Thiago Nascimento Rodrigues   Brentford     11        20.8
                 Antoine Semenyo Bournemouth      9        18.6
                    Hugo Ekitiké   Liverpool      8        17.9
                      Phil Foden    Man City      7        16.7
--- Playmaker: top 5 projected after GW 19
                  Name       Team  Assists  Proj_Assists
          Rayan Cherki   Man City        8   

### Your award picks

In [4]:
my_awards = {
    "Player of the Season": "Mohamed Salah",
    "Golden Boot": "Erling Haaland",
    "Playmaker": "Bukayo Saka",
    "Golden Glove": "David Raya Martín",
}

### Reveal

In [5]:
season_players = pgw[pgw["Season"] == SEASON].drop_duplicates("Player_ID")
names = dict(zip(my_awards, (season_players.set_index("Player_ID").loc[i, "Name"]
                             for i in pl.resolve_players(season_players, list(my_awards.values())))))

machine = pl.machine_award_picks(proj)
winners = pl.season_winners(pgw, SEASON)
rows = [{"Award": a, "You": names[a], "Machine": machine[a], "Winner": " / ".join(sorted(winners[a])),
         "Your pts": 5 * (names[a] in winners[a]), "Machine pts": 5 * (machine[a] in winners[a])} for a in pl.AWARDS]
reveal = pd.DataFrame(rows)
print(f"{SEASON} awards:  {PLAYER} {reveal['Your pts'].sum()} pts  |  Machine {reveal['Machine pts'].sum()} pts")
reveal

2025-26 awards:  You 10 pts  |  Machine 15 pts


,Award,You,Machine,Winner,Your pts,Machine pts
0,Player of the Season,Mohamed Salah,Erling Haaland,Erling Haaland,0,5
1,Golden Boot,Erling Haaland,Erling Haaland,Erling Haaland,5,5
2,Playmaker,Bukayo Saka,Rayan Cherki,Bruno Borges Fernandes,0,0
3,Golden Glove,David Raya Martín,David Raya Martín,David Raya Martín,5,5


In [6]:
log = pd.concat([
    pd.DataFrame({"Season": SEASON, "After_GW": AFTER_GW, "Award": reveal["Award"], "Player": PLAYER,
                  "Pick": reveal["You"], "Points": reveal["Your pts"]}),
    pd.DataFrame({"Season": SEASON, "After_GW": AFTER_GW, "Award": reveal["Award"], "Player": "Machine",
                  "Pick": reveal["Machine"], "Points": reveal["Machine pts"]}),
])
history = pl.save_results(log, "awards.csv", keys=["Season", "After_GW", "Award", "Player"])
print(history.groupby("Player")["Points"].sum().to_dict())

{'Machine': 15, 'You': 10}


## Player of the Week

Pick who you think will score the most FPL points in `POTW_GW`. The machine picks the player with the highest
predicted points from the fantasy model in notebook 05.

In [7]:
POTW_GW = AFTER_GW + 1
my_potw = "Erling Haaland"

model = pl.fit_player_model(pgw, SEASON)
season_rows = pgw[pgw["Season"] == SEASON].copy()
season_rows["Pred"] = model.predict(season_rows[pl.PLAYER_FEATURES])

def potw(gw):
    g = season_rows[season_rows["GW"] == gw]
    top = g["Points"].max()
    return g.sort_values("Pred", ascending=False).iloc[0], set(g.loc[g["Points"] == top, "Name"]), g

machine_pick, potw_winners, g = potw(POTW_GW)
my_pick = g.set_index("Player_ID").loc[pl.resolve_players(g, [my_potw])[0]]
print(f"GW {POTW_GW} Player of the Week: {' / '.join(potw_winners)} ({g['Points'].max()} pts)")
print(f"  {PLAYER}: {my_pick['Name']} scored {my_pick['Points']} -> {5 * (my_pick['Name'] in potw_winners)} pts")
print(f"  Machine: {machine_pick['Name']} scored {machine_pick['Points']} -> {5 * (machine_pick['Name'] in potw_winners)} pts")

GW 20 Player of the Week: Igor Thiago Nascimento Rodrigues / Malick Thiaw / Declan Rice (17 pts)
  You: Erling Haaland scored 2 -> 0 pts
  Machine: Erling Haaland scored 2 -> 0 pts


In [8]:
# How often does the machine's pick win, or at least finish in the week's top 10?
rows = []
for gw in sorted(season_rows["GW"].unique()):
    pick, win, g = potw(gw)
    rank = (g["Points"] > pick["Points"]).sum() + 1
    rows.append({"GW": gw, "Machine pick": pick["Name"], "Pick pts": pick["Points"],
                 "Winner": " / ".join(win), "Winner pts": g["Points"].max(), "Pick rank": rank})
potw_table = pd.DataFrame(rows)
print(f"Machine picked the Player of the Week in {(potw_table['Pick rank'] == 1).sum()} of {len(potw_table)} gameweeks; "
      f"top 10 in {(potw_table['Pick rank'] <= 10).sum()}")
potw_table.head(10)

Machine picked the Player of the Week in 0 of 38 gameweeks; top 10 in 10


,GW,Machine pick,Pick pts,Winner,Winner pts,Pick rank
0,1,Jarrod Bowen,2,Daniel Ballard,17,127
1,2,Daniel Ballard,1,Jurriën Timber,24,173
2,3,Bryan Mbeumo,10,Marc Guéhi,15,9
3,4,Mohamed Salah,9,Martín Zubimendi Ibáñez,16,13
4,5,Mohamed Salah,5,Ryan Gravenberch / Anton Stach,15,44
5,6,Antoine Semenyo,7,Omar Alderete,17,24
6,7,Erling Haaland,8,Antoine Semenyo,18,20
7,8,Erling Haaland,13,Reece James,18,6
8,9,Erling Haaland,2,Micky van de Ven,23,104
9,10,Mohamed Salah,10,Diego Gómez Amarilla,16,8


## Backtest: how often is the machine right at halfway?

For every playable season, the machine makes its award picks after gameweek 19, using only that season's first
19 gameweeks and models trained on earlier seasons. As a sanity check, a projection made after gameweek 38 must equal
the real season totals.

In [9]:
rows = []
for season in pl.PLAYABLE_SEASONS:
    gm = pl.fit_goal_models(mdf[mdf["Season"] < season])
    picks = pl.machine_award_picks(pl.award_projections(pgw, fixtures, team_games, elo_hist, gm, season, 19))
    win = pl.season_winners(pgw, season)
    rows.append({"Season": season, **{a: ("✔ " if picks[a] in win[a] else "✘ ") + picks[a] for a in pl.AWARDS}})

    end = pl.award_projections(pgw, fixtures, team_games, elo_hist, gm, season, 38)
    assert pl.machine_award_picks(end)["Golden Boot"] in win["Golden Boot"]
    assert pl.machine_award_picks(end)["Golden Glove"] in win["Golden Glove"]
backtest = pd.DataFrame(rows)
hits = {a: backtest[a].str.startswith("✔").sum() for a in pl.AWARDS}
print("Correct at GW 19 (out of", len(backtest), "seasons):", hits)
backtest

Correct at GW 19 (out of 9 seasons): {'Player of the Season': np.int64(6), 'Golden Boot': np.int64(8), 'Playmaker': np.int64(3), 'Golden Glove': np.int64(6)}


,Season,Player of the Season,Golden Boot,Playmaker,Golden Glove
0,2017-18,✔ Mohamed Salah,✔ Mohamed Salah,✘ Leroy Sané,✘ Thibaut Courtois
1,2018-19,✔ Mohamed Salah,✔ Mohamed Salah,✘ Leroy Sané,✔ Alisson Ramses Becker
2,2019-20,✘ Jamie Vardy,✔ Jamie Vardy,✔ Kevin De Bruyne,✔ Ederson Santana de Moraes
3,2020-21,✘ Harry Kane,✘ Mohamed Salah,✘ Kevin De Bruyne,✔ Ederson Santana de Moraes
4,2021-22,✔ Mohamed Salah,✔ Mohamed Salah,✘ Mohamed Salah,✔ Ederson Santana de Moraes
5,2022-23,✔ Erling Haaland,✔ Erling Haaland,✔ Kevin De Bruyne,✘ Nick Pope
6,2023-24,✘ Mohamed Salah,✔ Erling Haaland,✘ Julián Álvarez,✘ Alisson Ramses Becker
7,2024-25,✔ Mohamed Salah,✔ Mohamed Salah,✔ Mohamed Salah,✔ David Raya Martin
8,2025-26,✔ Erling Haaland,✔ Erling Haaland,✘ Rayan Cherki,✔ David Raya Martín
